# [6.2] Gemma Scope Deep Dive - Exercises

Gemma Scope gives us released sparse features for Gemma-family models, but a released feature is only a coordinate system. In this notebook you build the validation ladder around one feature hypothesis: metadata first, prompt-level scores, held-out validation, base-vs-instruction deltas, causal controls, steering guards, and direct logit attribution.

```yaml
gt_tier: GT-1 artifact preflight with GT-0 validation controls
exercise_id: 6_2_gemma_scope_deep_dive
expected_runtime: 45-75 minutes for CPU exercises; several minutes for CUDA artifact and activation preflight
requires_gpu: true for the released-artifact preflight; false for the implementation exercises
```

<details>
<summary>Expected output</summary>

By the end, every local test should print an "All tests ... passed" line, and the final report-backed cells should show the pinned Gemma Scope artifact, Gemma 3 layer-13 activation shapes, held-out AUC controls, and peak VRAM.

</details>

<details>
<summary>Help - how to read this section</summary>

Use the original ARENA loop: look at a feature hypothesis, implement the measurement, validate it against controls, then state the claim boundary. A tag or dashboard label is a starting hypothesis, not the conclusion.

</details>


In [ ]:
import json
import sys
from dataclasses import dataclass
from pathlib import Path
from typing import Literal

import torch as t

chapter = "chapter6_sparse_feature_methods"
section = "part2_gemma_scope_deep_dive"
root_dir = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / chapter).exists())
exercises_dir = root_dir / chapter / "exercises"
section_dir = exercises_dir / section
if str(root_dir) not in sys.path:
    sys.path.append(str(root_dir))
if str(exercises_dir) not in sys.path:
    sys.path.append(str(exercises_dir))

import part2_gemma_scope_deep_dive.tests as tests


@dataclass(frozen=True)
class FeatureArtifactMetadata:
    model_name: str
    artifact_name: str
    artifact_type: Literal["sae", "transcoder"]
    layer: int
    hook_name: str
    d_model: int
    n_features: int


@dataclass(frozen=True)
class TaggedFeatureSpec:
    feature_id: int
    layer: int
    tags: tuple[str, ...]
    description: str


@dataclass(frozen=True)
class FeatureValidationSuiteReport:
    feature_auc: float
    baseline_auc: float
    auc_margin: float
    threshold_accuracy: float
    positive_mean: float
    negative_mean: float
    passes_baseline: bool


@dataclass(frozen=True)
class BaseInstructionFeatureDelta:
    base_mean: float
    instruction_mean: float
    delta: float
    abs_delta: float


@dataclass(frozen=True)
class AblationControlReport:
    baseline_mean: float
    ablated_mean: float
    random_ablated_mean: float
    ablation_delta: float
    random_delta: float
    passes_control: bool


@dataclass(frozen=True)
class SteeringSafetyReport:
    baseline_mean: float
    steered_mean: float
    random_mean: float
    steered_delta: float
    random_delta: float
    perplexity_ratio: float
    passes_control: bool
    passes_perplexity_guard: bool


## 1. Artifact Metadata And Tags

Every feature report should be reloadable. The model name, artifact name, hook point, layer, dimensionality, and feature count are not decoration; they define what coordinate system the feature lives in.

<details>
<summary>Expected output</summary>

Complete metadata should pass, missing model names should fail, and tag selection should preserve feature order. When correct, the test prints:

```text
All tests in `test_metadata_completeness_and_tag_selection` passed!
```

</details>

<details>
<summary>Help - descriptions are not evidence</summary>

A description such as "refusal phrase feature" is a hypothesis label. It becomes evidence only after held-out positives, matched negatives, and controls support it.

</details>


In [ ]:
def metadata_is_complete(metadata: FeatureArtifactMetadata) -> bool:
    raise NotImplementedError()


def features_with_tag(
    features: list[TaggedFeatureSpec],
    tag: str,
) -> list[TaggedFeatureSpec]:
    raise NotImplementedError()


tests.test_metadata_completeness_and_tag_selection(
    FeatureArtifactMetadata,
    TaggedFeatureSpec,
    metadata_is_complete,
    features_with_tag,
)


## 2. Prompt-Level Feature Scores

SAE activations are usually per token. Before you claim a prompt activates a feature, you need a reduction rule: max, mean, or last-token score.

<details>
<summary>Expected output</summary>

For the controlled tensor, max, mean, and last-token reductions should match the independent reference. Rank-2 activations should be treated as already reduced. When correct, the test prints:

```text
All tests in `test_feature_score_vector_reductions_match_reference` passed!
```

</details>

<details>
<summary>Help - report the reduction rule</summary>

Max pooling can make a feature look more selective than mean pooling. A validation result without its reduction rule is not reproducible.

</details>


In [ ]:
def feature_score_vector(
    feature_acts: t.Tensor,
    feature_id: int,
    *,
    reduction: Literal["max", "mean", "last"] = "max",
) -> t.Tensor:
    raise NotImplementedError()


tests.test_feature_score_vector_reductions_match_reference(feature_score_vector)


## 3. Held-Out Feature Validation

Top activating examples answer "what can make this coordinate large?" They do not answer whether the feature separates a concept from nearby non-concepts. Here you implement AUC and a compact validation report.

<details>
<summary>Expected output</summary>

The candidate feature should have AUC `1.0`, the inverted baseline should have AUC `0.0`, the AUC margin should be `1.0`, and positive examples should have larger mean score than negatives. When correct, the test prints:

```text
All tests in `test_validate_feature_scores_beats_baseline_and_reports_means` passed!
```

</details>

<details>
<summary>Help - top activations need matched negatives</summary>

A feature that fires on technical text might also fire on any long prompt, any markdown list, or any rare-token sequence. Matched negatives and baseline features remove the easiest false explanations.

</details>


In [ ]:
def roc_auc_binary(scores: t.Tensor, labels: t.Tensor) -> float:
    raise NotImplementedError()


def validate_feature_scores(
    feature_scores: t.Tensor,
    labels: t.Tensor,
    baseline_scores: t.Tensor,
    *,
    min_auc_margin: float = 0.1,
) -> FeatureValidationSuiteReport:
    raise NotImplementedError()


tests.test_validate_feature_scores_beats_baseline_and_reports_means(
    validate_feature_scores,
    roc_auc_binary,
)


## 4. Base Vs Instruction Deltas

Instruction tuning can change which sparse features are active, but a delta is still a correlation. Preserve the sign before you rank by magnitude.

<details>
<summary>Expected output</summary>

The test checks both positive and negative deltas. A base mean of `0.2` and instruction mean of `0.4` should give signed delta `0.2` and absolute delta `0.2`. When correct, the test prints:

```text
All tests in `test_base_instruction_delta_reports_signed_and_abs_change` passed!
```

</details>

<details>
<summary>Help - sign is part of the observation</summary>

A feature that increases after instruction tuning suggests a different hypothesis from one that decreases. Taking `abs` too early throws away that information.

</details>


In [ ]:
def base_instruction_feature_delta(
    base_scores: t.Tensor,
    instruction_scores: t.Tensor,
) -> BaseInstructionFeatureDelta:
    raise NotImplementedError()


tests.test_base_instruction_delta_reports_signed_and_abs_change(
    base_instruction_feature_delta,
)


## 5. Ablation Control

If ablating a feature changes a behavior, you still need to know whether a matched random-feature ablation changes it just as much. The control is the point of the exercise.

<details>
<summary>Expected output</summary>

Target ablation should reduce the score by `0.75`, random ablation by `0.15`, and the report should fail when those roles are reversed. When correct, the test prints:

```text
All tests in `test_ablation_control_requires_target_ablation_to_beat_random` passed!
```

</details>

<details>
<summary>Help - any direction can change something</summary>

High-dimensional residual streams are easy to perturb. A target ablation becomes evidence only when it beats a matched random control on the same metric.

</details>


In [ ]:
def ablation_control_report(
    baseline_scores: t.Tensor,
    ablated_scores: t.Tensor,
    random_ablated_scores: t.Tensor,
) -> AblationControlReport:
    raise NotImplementedError()


tests.test_ablation_control_requires_target_ablation_to_beat_random(
    ablation_control_report,
)


## 6. Steering Guard

Steering should move the target score more than a random control without wrecking general model quality. This toy guard uses a perplexity ratio as the coarse safety check.

<details>
<summary>Expected output</summary>

Useful steering should pass when the perplexity ratio is `1.1`, and fail the guard when the same target effect comes with a ratio of `1.5`. When correct, the test prints:

```text
All tests in `test_steering_safety_report_checks_control_and_perplexity_guard` passed!
```

</details>

<details>
<summary>Help - steering can break the model</summary>

A giant vector can make almost any score move. A useful steering result needs a target effect, a random-direction control, and a degradation guard.

</details>


In [ ]:
def steering_safety_report(
    baseline_scores: t.Tensor,
    steered_scores: t.Tensor,
    random_control_scores: t.Tensor,
    *,
    baseline_perplexity: float,
    steered_perplexity: float,
    max_perplexity_ratio: float = 1.2,
) -> SteeringSafetyReport:
    raise NotImplementedError()


tests.test_steering_safety_report_checks_control_and_perplexity_guard(
    steering_safety_report,
)


## 7. Direct Logit Attribution

Direct logit attribution projects decoder vectors through the unembedding. It is useful for generating token-level hypotheses, but it is not causal proof.

<details>
<summary>Expected output</summary>

Selecting token ids `[0, 2]` should return `[[1.0, 3.0], [4.0, 6.0]]` for the controlled decoder/unembedding pair. When correct, the test prints:

```text
All tests in `test_direct_logit_attribution_matches_selected_token_projection` passed!
```

</details>

<details>
<summary>Help - DLA is a hypothesis generator</summary>

DLA tells you which vocabulary directions a decoder vector points toward. It does not prove the model uses that feature in the behavior you care about.

</details>


In [ ]:
def direct_logit_attribution(
    decoder_vectors: t.Tensor,
    unembedding: t.Tensor,
    token_ids: t.Tensor | list[int] | None = None,
) -> t.Tensor:
    raise NotImplementedError()


tests.test_direct_logit_attribution_matches_selected_token_projection(
    direct_logit_attribution,
)


## Whole-Notebook Contract

Once all exercises pass, your implementation should satisfy the same local smoke-test contract as `solutions.py`.

<details>
<summary>Expected output</summary>

After uncommenting the last line, the test should print:

```text
All tests in `test_notebook_contract` passed!
```

</details>


In [ ]:
def run_smoke_test(cpu: bool = True) -> dict:
    _ = cpu
    scores = t.tensor([0.1, 0.2, 0.9, 1.0])
    labels = t.tensor([0, 0, 1, 1], dtype=t.bool)
    baseline_scores = t.tensor([1.0, 0.9, 0.2, 0.1])
    base = t.tensor([0.1, 0.2, 0.3])
    instruction = t.tensor([0.3, 0.4, 0.5])
    decoder_vectors = t.tensor([[1.0, 0.0], [0.0, 1.0]])
    unembedding = t.tensor([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]])
    return {
        "metadata": {"metadata_complete": metadata_is_complete(FeatureArtifactMetadata(
            "gemma-test", "layer_0_resid_sae", "sae", 0, "resid_post", 4, 8
        ))},
        "validation": validate_feature_scores(scores, labels, baseline_scores).__dict__,
        "base_instruction_delta": base_instruction_feature_delta(base, instruction).__dict__,
        "ablation": ablation_control_report(
            t.tensor([1.0, 1.0]),
            t.tensor([0.2, 0.3]),
            t.tensor([0.8, 0.9]),
        ).__dict__,
        "steering": steering_safety_report(
            t.tensor([0.1, 0.2]),
            t.tensor([0.6, 0.7]),
            t.tensor([0.2, 0.3]),
            baseline_perplexity=10.0,
            steered_perplexity=11.0,
        ).__dict__,
        "logit_attribution": direct_logit_attribution(
            decoder_vectors,
            unembedding,
            token_ids=[0, 2],
        ).tolist(),
    }


# Uncomment after finishing all exercises.
# tests.test_notebook_contract(run_smoke_test)


## Signature Result

The final result is report-backed and uses the committed CUDA evidence. It does not rerun the heavy Gemma Scope path inside the notebook; rerun `solutions.run_gpu_test(max_vram_gb=24.0)` from Python when you want to refresh the report.

<details>
<summary>Expected output</summary>

The table should show `google/gemma-scope-2-1b-it`, artifact `resid_post/layer_13_width_16k_l0_small`, width `16384`, `d_model=1152`, selected/random features `15586 / 7121`, held-out AUC `1.000`, random-feature baseline AUC `0.500`, label-shuffle AUC `0.000`, and peak VRAM around `2.236 GB`.

</details>

<details>
<summary>Interpreting the signature result</summary>

This validates one benign technical-vs-narrative feature hypothesis on authenticated Gemma 3 layer-13 activations. It does not validate broad Gemma Scope semantics or real-model causal steering.

</details>


In [ ]:
def _load_committed_gpu_report() -> dict:
    report = json.loads((section_dir / "verification_report.json").read_text())
    assert report["accepted"] and report["tests_passed"]
    gpu = report["metrics"]["gpu_test"]
    assert gpu["cuda_available"]
    assert gpu["within_vram_budget"]
    return gpu


def run_gpu_test(max_vram_gb: float = 24.0) -> dict:
    gpu = _load_committed_gpu_report()
    assert gpu["peak_vram_gb"] <= max_vram_gb
    return gpu


def run_full_experiment(max_vram_gb: float = 24.0) -> dict:
    return run_gpu_test(max_vram_gb=max_vram_gb)


def signature_table(gpu: dict) -> list[tuple[str, object]]:
    real = gpu["gemma_scope_real_activation_preflight"]
    return [
        ("Gemma Scope repo", gpu["gemma_scope_repo_id"]),
        ("artifact", gpu["gemma_scope_artifact_path"]),
        ("revision", gpu["gemma_scope_revision"][:12]),
        ("shape", f"d_model {gpu['gemma_scope_d_model']}, width {gpu['gemma_scope_width']}"),
        ("encoder / decoder", f"{gpu['gemma_scope_w_enc_shape']} / {gpu['gemma_scope_w_dec_shape']}"),
        ("CUDA forward", gpu["gemma_scope_forward_passed"]),
        ("base model authenticated", gpu["gemma3_base_authenticated"]),
        ("train / held-out prompts", f"{real['train_prompt_count']} / {real['heldout_prompt_count']}"),
        ("selected / random feature", f"{real['selected_feature_id']} / {real['random_control_feature_id']}"),
        ("held-out AUC", round(real["feature_auc"], 3)),
        ("random-feature AUC", round(real["baseline_auc"], 3)),
        ("label-shuffle AUC", round(real["label_shuffle_auc"], 3)),
        ("positive / negative mean", f"{real['positive_mean']:.2f} / {real['negative_mean']:.2f}"),
        ("peak VRAM GB", round(gpu["peak_vram_gb"], 3)),
    ]


gpu = run_gpu_test(max_vram_gb=24.0)
signature_table(gpu)


In [ ]:
import matplotlib.pyplot as plt

gpu = run_gpu_test(max_vram_gb=24.0)
real = gpu["gemma_scope_real_activation_preflight"]
fig, axes = plt.subplots(1, 3, figsize=(12, 3.2))

axes[0].bar(
    ["feature", "random", "shuffle"],
    [real["feature_auc"], real["baseline_auc"], real["label_shuffle_auc"]],
    color=["#2563eb", "#94a3b8", "#f97316"],
)
axes[0].set_ylim(0, 1.05)
axes[0].set_title("Held-out AUC controls")
axes[0].set_ylabel("AUC")

axes[1].bar(
    ["technical", "narrative"],
    [real["positive_mean"], real["negative_mean"]],
    color=["#16a34a", "#94a3b8"],
)
axes[1].set_title(f"Feature {real['selected_feature_id']} scores")
axes[1].set_ylabel("mean activation")

axes[2].bar(
    ["artifact", "real activations"],
    [gpu["gemma_scope_peak_vram_gb"], real["peak_vram_gb"]],
    color=["#7c3aed", "#0f766e"],
)
axes[2].set_title("Peak CUDA memory")
axes[2].set_ylabel("GB")

fig.tight_layout()
plt.show()


## Limitations

The local tests use tiny tensors and prove the exercise contract, not real-model semantics. The committed GPU report proves a scoped Gemma Scope artifact and activation path: one pinned artifact, one Gemma 3 layer, one benign technical-vs-narrative split, one selected feature, and explicit controls. It does not prove broad Gemma Scope coverage, refusal-feature semantics, safety-relevant steering, or causal ablation on real Gemma activations.
